In [4]:
# experiments/copy_task.py
# ============================================================
# Delayed Copy Task — Axiom vs LSTM
#
# Reproduces Table 1 from the paper.
# Usage:
#   python copy_task.py --delay 100 --model axiom
#   python copy_task.py --delay 100 --model lstm
#   python copy_task.py --delay 300 --model axiom
#
# Expected results (from paper):
#   Axiom delay=100: ~99.9%
#   LSTM  delay=100: ~13.5% (random baseline = 12.5%)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import argparse
import time

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# TASK
# ============================================================

def make_batch(batch_size, seq_len, delay, vocab_size=8, device='cpu'):
    """
    Generate a delayed copy task batch.

    Structure:
        [data: seq_len] [blank: delay] [marker: 1] [zeros: seq_len]
    Target:
        [-1: seq_len+delay+1] [data: seq_len]   (-1 = ignore in loss)

    The model must reproduce `data` exactly in the output positions.
    Random baseline: 12.5% (1/8 for vocab_size=8)
    """
    total  = seq_len + delay + 1 + seq_len
    data   = torch.randint(0, vocab_size, (batch_size, seq_len))
    blank  = torch.full((batch_size, delay), vocab_size)
    marker = torch.full((batch_size, 1), vocab_size + 1)
    zeros  = torch.zeros(batch_size, seq_len, dtype=torch.long)

    x   = torch.cat([data, blank, marker, zeros], dim=1).to(device)
    tgt = torch.cat([
        torch.full((batch_size, seq_len + delay + 1), -1),
        data
    ], dim=1).to(device)
    return x, tgt

def onehot(x, num_classes, device):
    out = torch.zeros(*x.shape, num_classes, device=device)
    out.scatter_(-1, x.unsqueeze(-1), 1.0)
    return out

# ============================================================
# MODELS
# ============================================================

class AxiomRNN(nn.Module):
    """
    Pure unitary RNN parameterized by Householder reflections.
    No forget gate — memory preserved by unitary guarantee.

    Hidden state: (batch, res, res) real matrix
    Update:       h_t = W^T @ h_{t-1} @ W + gate_t * val_t
    """
    def __init__(self, input_size, res, rank, output_size):
        super().__init__()
        self.res  = res
        self.rank = rank
        self.input_proj = nn.Linear(input_size, res * res)
        self.input_gate = nn.Linear(input_size, res * res)
        self.v          = nn.Parameter(torch.randn(rank, res) * 0.02)
        self.ln         = nn.LayerNorm(res * res)
        self.head       = nn.Linear(res * res, output_size)
        self.h0         = nn.Parameter(torch.zeros(res, res))

    def _get_rotation(self):
        """Build unitary matrix from Householder reflections."""
        v_n   = self.v / (torch.norm(self.v, dim=1, keepdim=True) + 1e-8)
        eye   = torch.eye(self.res, device=self.v.device)
        outer = torch.bmm(v_n.unsqueeze(2), v_n.unsqueeze(1))
        H_all = eye.unsqueeze(0) - 2.0 * outer   # (rank, res, res)
        W = H_all[0]
        for i in range(1, self.rank):
            W = torch.matmul(W, H_all[i])
        return W  # unitary: W^T W = I

    def forward_step(self, x, h):
        """Single timestep. x: (batch, input_size), h: (batch, res, res)"""
        batch  = x.shape[0]
        W      = self._get_rotation()
        val    = torch.tanh(self.input_proj(x)).view(batch, self.res, self.res)
        gate   = torch.sigmoid(self.input_gate(x)).view(batch, self.res, self.res)
        # Unitary rotation — no forget gate, no memory decay
        h_next = torch.matmul(torch.matmul(W.t(), h), W) + gate * val
        out    = self.head(self.ln(h_next.reshape(batch, -1)))
        return out, h_next

    def init_hidden(self, batch):
        return self.h0.unsqueeze(0).expand(batch, -1, -1).contiguous()


class VanillaLSTM(nn.Module):
    """Standard LSTM baseline."""
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=False)
        self.head = nn.Linear(hidden_size, output_size)

    def forward_step(self, x, state):
        """x: (batch, input_size), state: (h, c) each (1, batch, hidden)"""
        out, state = self.lstm(x.unsqueeze(0), state)
        return self.head(out.squeeze(0)), state

    def init_hidden(self, batch, device):
        h = torch.zeros(1, batch, self.hidden_size, device=device)
        c = torch.zeros(1, batch, self.hidden_size, device=device)
        return (h, c)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


# ============================================================
# TRAINING
# ============================================================

def train_and_eval(model_name, delay, epochs=40, batch_size=128,
                   seq_len=10, vocab_size=8, steps_per_epoch=100):
    device    = get_device()
    total_len = seq_len + delay + 1 + seq_len
    in_size   = vocab_size + 2  # noise + blank + marker

    print(f"\nModel: {model_name.upper()}")
    print(f"Delay: {delay} | Total sequence: {total_len} steps")

    if model_name == 'axiom':
        model = AxiomRNN(in_size, res=16, rank=8,
                          output_size=vocab_size).to(device)
    elif model_name == 'lstm':
        model = VanillaLSTM(in_size, hidden_size=160,
                             output_size=vocab_size).to(device)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    print(f"Parameters: {count_params(model):,}")

    opt  = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss(ignore_index=-1)
    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for _ in range(steps_per_epoch):
            x, tgt = make_batch(batch_size, seq_len, delay,
                                  vocab_size, device)
            x_oh = onehot(x, in_size, device)   # (B, T, V)
            opt.zero_grad()

            # Run model step by step
            if model_name == 'axiom':
                h = model.init_hidden(batch_size).to(device)
                logits_all = []
                for t in range(total_len):
                    out, h = model.forward_step(x_oh[:, t, :], h)
                    logits_all.append(out)
                logits = torch.stack(logits_all, 1)          # (B, T, V)
            else:
                state = model.init_hidden(batch_size, device)
                logits_all = []
                for t in range(total_len):
                    out, state = model.forward_step(x_oh[:, t, :], state)
                    logits_all.append(out)
                logits = torch.stack(logits_all, 1)

            loss = crit(logits.reshape(-1, vocab_size), tgt.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            epoch_loss += loss.item()

        # Evaluate
        model.eval()
        with torch.no_grad():
            x, tgt = make_batch(512, seq_len, delay, vocab_size, device)
            x_oh   = onehot(x, in_size, device)
            mask   = (tgt != -1)

            if model_name == 'axiom':
                h = model.init_hidden(512).to(device)
                logits_all = []
                for t in range(total_len):
                    out, h = model.forward_step(x_oh[:, t, :], h)
                    logits_all.append(out)
            else:
                state = model.init_hidden(512, device)
                logits_all = []
                for t in range(total_len):
                    out, state = model.forward_step(x_oh[:, t, :], state)
                    logits_all.append(out)

            logits = torch.stack(logits_all, 1)
            preds  = logits.argmax(-1)
            acc    = (preds[mask] == tgt[mask]).float().mean().item() * 100
            best_acc = max(best_acc, acc)

        if (epoch + 1) % 10 == 0 or acc > 95:
            print(f"  Epoch {epoch+1:2d}: {acc:.1f}%  "
                  f"(loss={epoch_loss/steps_per_epoch:.3f})")
            if acc >= 99.5:
                print(f"  Converged at epoch {epoch+1}!")
                break

    print(f"\n  >>> Best accuracy: {best_acc:.1f}%")
    print(f"  >>> Random baseline: {100/vocab_size:.1f}%")
    return best_acc


# ============================================================
# MAIN
# ============================================================

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Delayed Copy Task')
    parser.add_argument('--model',  default='both',
                        choices=['axiom', 'lstm', 'both'],
                        help='Model to train')
    parser.add_argument('--delay',  type=int, default=100,
                        help='Delay length (100, 300, or 500)')
    parser.add_argument('--epochs', type=int, default=80)
    args = parser.parse_args([])

    print("=" * 55)
    print("DELAYED COPY TASK")
    print(f"Delay={args.delay} | Random baseline=12.5%")
    print("=" * 55)

    results = {}
    models  = ['axiom', 'lstm'] if args.model == 'both' else [args.model]

    for m in models:
        results[m] = train_and_eval(m, args.delay, epochs=args.epochs)

    if len(results) == 2:
        print("\n" + "=" * 55)
        print("COMPARISON")
        print("=" * 55)
        print(f"Axiom: {results['axiom']:.1f}%")
        print(f"LSTM:  {results['lstm']:.1f}%")
        print(f"Gap:   {results['axiom'] - results['lstm']:+.1f}%")
        print(f"\nExpected from paper:")
        print(f"  Axiom delay=100: ~99.9%")
        print(f"  LSTM  delay=100: ~13.5%")

DELAYED COPY TASK
Delay=100 | Random baseline=12.5%

Model: AXIOM
Delay: 100 | Total sequence: 121 steps
Parameters: 8,584
  Epoch 10: 52.7%  (loss=1.333)
  Epoch 20: 70.3%  (loss=1.085)
  Epoch 30: 87.3%  (loss=0.486)
  Epoch 34: 95.3%  (loss=0.292)
  Epoch 35: 95.2%  (loss=0.257)
  Epoch 36: 96.7%  (loss=0.231)
  Epoch 37: 98.0%  (loss=0.206)
  Epoch 40: 83.9%  (loss=0.582)
  Epoch 47: 96.2%  (loss=0.390)
  Epoch 50: 41.6%  (loss=0.925)
  Epoch 54: 95.3%  (loss=0.564)
  Epoch 56: 97.9%  (loss=0.658)
  Epoch 60: 70.9%  (loss=0.728)
  Epoch 62: 96.9%  (loss=0.160)
  Epoch 64: 95.2%  (loss=0.116)
  Epoch 65: 98.4%  (loss=0.101)
  Epoch 67: 99.8%  (loss=0.678)
  Converged at epoch 67!

  >>> Best accuracy: 99.8%
  >>> Random baseline: 12.5%

Model: LSTM
Delay: 100 | Total sequence: 121 steps
Parameters: 111,368
  Epoch 10: 12.4%  (loss=2.079)
  Epoch 20: 12.4%  (loss=2.079)
  Epoch 30: 12.6%  (loss=2.079)
  Epoch 40: 12.7%  (loss=2.079)
  Epoch 50: 11.8%  (loss=2.079)
  Epoch 60: 12.2%  